# RSNN on FI-2010 LOB — Extended Experiments

**Carried-forward results** (from fi-2010 notebook):
| Model | Test Accuracy |
|---|---|
| RSNN Direct Input | 37.56% |
| RSNN Delta Modulation | 27.53% |
| RSNN Temporal Contrast (th=0.01) | 29.33% |
| RSNN Signed Contrast (th=0.01) | 27.56% |
| Graded Spike RSNN | 27.53% |
| LSTM (2L, 128) | **66.07%** |
| 1D-CNN | 64.65% |

**Split**: DeepLOB convention — Days 1-7 train (80/20 train/val split),
Day 8+9 test. Early stopping on validation.

**Narrative**: The RSNN fundamentally fails on LOB data (28.5pp gap to LSTM).
This notebook diagnoses why and tests targeted fixes.

**Experiments**:
- F1-F3: Encoding fixes (adaptive delta, spike diagnosis, log-scale)
- F4-F6: Architecture (tau sweep, hidden size, input BN)
- F7-F10: Analysis suite (noise, efficiency, ablation, early classification)
- F11-F13: Domain-specific (multi-horizon, price vs volume, per-level)

**Dataset**: Add 'fi-2010' by ulfricirons to Kaggle.

## 0. Setup

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'snntorch', '-q'])
print('Done.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 3.3 MB/s eta 0:00:00


Done.


In [2]:
import os, glob, copy, time, json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, f1_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

SAVE_DIR = '/kaggle/working'
RESULTS = {}

Device: cuda


## 1. Spike Encoding (including adaptive delta)

In [3]:
class SpikeEncoder:
    @staticmethod
    def direct(signal):
        return signal

    @staticmethod
    def delta_modulation(signal, threshold=0.1):
        T, C = signal.shape
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > threshold
            down = diff < -threshold
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes

    @staticmethod
    def adaptive_delta(signal, percentile=95):
        """Per-channel adaptive threshold. Amir et al. 2017 CVPR.
        Critical for LOB: fixed thresholds produce zero spikes because
        inter-step changes are O(1e-4)."""
        T, C = signal.shape
        diffs = np.abs(np.diff(signal, axis=0))
        thresholds = np.percentile(diffs, percentile, axis=0)
        thresholds = np.maximum(thresholds, 1e-8)
        spikes = np.zeros_like(signal)
        reference = signal[0].copy()
        for t in range(1, T):
            diff = signal[t] - reference
            up = diff > thresholds
            down = diff < -thresholds
            spikes[t, up] = 1.0
            spikes[t, down] = -1.0
            reference[up] = signal[t, up]
            reference[down] = signal[t, down]
        return spikes, thresholds

    @staticmethod
    def log_scale(signal):
        """Log-scale amplification of small changes.
        sign(x) * log(1 + |x|). Stretches the small-diff region."""
        return np.sign(signal) * np.log1p(np.abs(signal))

## 2. SNN Components

Identical LIF + Readout + SNN from SHD. Also LSTM and CNN baselines.

In [4]:
class SurrogateSpike(torch.autograd.Function):
    beta = 40.0
    @staticmethod
    def forward(ctx, mem, threshold=1.0):
        ctx.save_for_backward(mem)
        ctx.threshold = threshold
        return (mem >= threshold).float()
    @staticmethod
    def backward(ctx, grad_output):
        mem, = ctx.saved_tensors
        v = mem - ctx.threshold
        grad = 1.0 / (1.0 + SurrogateSpike.beta * torch.abs(v)) ** 2
        return grad_output * grad, None

def spike_fn(x, threshold=1.0):
    return SurrogateSpike.apply(x, threshold)

In [5]:
class LIFLayer(nn.Module):
    def __init__(self, input_size, hidden_size, recurrent=False,
                 tau_mem_init=20.0, tau_syn_init=10.0, dt=10.0,
                 learnable_tau=False, dropout=0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.recurrent = recurrent
        self.dt = dt
        self.dropout = dropout
        self.W_ff = nn.Linear(input_size, hidden_size, bias=False)
        if recurrent:
            self.W_rec = nn.Linear(hidden_size, hidden_size, bias=False)
        if learnable_tau:
            self.log_tau_mem = nn.Parameter(torch.tensor(np.log(tau_mem_init)))
            self.log_tau_syn = nn.Parameter(torch.tensor(np.log(tau_syn_init)))
        else:
            self.register_buffer('log_tau_mem', torch.tensor(np.log(tau_mem_init)))
            self.register_buffer('log_tau_syn', torch.tensor(np.log(tau_syn_init)))
        nn.init.kaiming_uniform_(self.W_ff.weight, nonlinearity='linear')
        if recurrent:
            nn.init.kaiming_uniform_(self.W_rec.weight, nonlinearity='linear')
    @property
    def alpha(self): return torch.exp(-self.dt / torch.exp(self.log_tau_syn))
    @property
    def beta(self): return torch.exp(-self.dt / torch.exp(self.log_tau_mem))
    def forward(self, x):
        B, T, _ = x.shape
        alpha, beta = self.alpha, self.beta
        syn = torch.zeros(B, self.hidden_size, device=x.device)
        mem = torch.zeros(B, self.hidden_size, device=x.device)
        prev_spk = torch.zeros(B, self.hidden_size, device=x.device)
        spike_rec, mem_rec = [], []
        for t in range(T):
            syn = alpha * syn + self.W_ff(x[:, t])
            if self.recurrent:
                rec_spk = F.dropout(prev_spk, p=self.dropout, training=self.training) if self.dropout > 0 else prev_spk
                syn = syn + self.W_rec(rec_spk)
            mem = beta * mem * (1.0 - prev_spk) + (1.0 - beta) * syn
            spk = spike_fn(mem, threshold=1.0)
            spike_rec.append(spk); mem_rec.append(mem); prev_spk = spk
        return torch.stack(spike_rec, dim=1), torch.stack(mem_rec, dim=1)

class ReadoutLayer(nn.Module):
    def __init__(self, input_size, output_size, tau_mem=20.0, dt=10.0):
        super().__init__()
        self.fc = nn.Linear(input_size, output_size, bias=False)
        self.beta = np.exp(-dt / tau_mem)
        nn.init.kaiming_uniform_(self.fc.weight, nonlinearity='linear')
    def forward(self, x):
        B, T, _ = x.shape
        mem = torch.zeros(B, self.fc.out_features, device=x.device)
        mem_rec = []
        for t in range(T):
            mem = self.beta * mem + (1.0 - self.beta) * self.fc(x[:, t])
            mem_rec.append(mem)
        return torch.stack(mem_rec, dim=1)

class SNN(nn.Module):
    def __init__(self, input_size, hidden_size=256, output_size=3,
                 n_hidden_layers=1, recurrent=True, tau_mem=20.0, tau_syn=10.0,
                 dt=10.0, learnable_tau=False, loss_mode='max_over_time', dropout=0.0):
        super().__init__()
        self.loss_mode = loss_mode; self.hidden_size = hidden_size
        layers = []
        for i in range(n_hidden_layers):
            in_sz = input_size if i == 0 else hidden_size
            layers.append(LIFLayer(in_sz, hidden_size, recurrent, tau_mem, tau_syn, dt, learnable_tau, dropout))
        self.hidden_layers = nn.ModuleList(layers)
        self.readout = ReadoutLayer(hidden_size, output_size, tau_mem, dt)
    def forward(self, x):
        all_spikes = []; h = x
        for layer in self.hidden_layers:
            spikes, _ = layer(h); all_spikes.append(spikes); h = spikes
        out_mem = self.readout(h)
        if self.loss_mode == 'max_over_time': output, _ = torch.max(out_mem, dim=1)
        elif self.loss_mode == 'last_timestep': output = out_mem[:, -1, :]
        else: raise ValueError(self.loss_mode)
        return output, all_spikes, out_mem
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

class LSTMBaseline(nn.Module):
    def __init__(self, input_size, hidden_size=128, n_layers=2, output_size=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    def forward(self, x): out, _ = self.lstm(x); return self.fc(out[:, -1, :])
    def forward_seq(self, x): out, _ = self.lstm(x); return out
    def count_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

class CNNBaseline(nn.Module):
    def __init__(self, input_channels, output_size=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_channels, 64, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(128, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(128, output_size)
    def forward(self, x): return self.fc(self.conv(x.transpose(1,2)).squeeze(-1))
    def count_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 3. Training Engine

Early stopping on validation (80/20 split from training data).
Matches DeepLOB protocol.

In [6]:
def spike_regularization(all_spikes, theta_l=0.01, s_l=1.0, theta_u=100.0, s_u=0.06):
    reg = torch.tensor(0.0, device=all_spikes[0].device)
    for spk in all_spikes:
        B, T, N = spk.shape
        mean_rate = spk.sum(dim=1) / T
        reg += s_l / (B * N) * (F.relu(theta_l - mean_rate) ** 2).sum()
        pop_count = spk.sum(dim=(1, 2)) / N
        reg += s_u / B * (F.relu(pop_count - theta_u) ** 2).sum()
    return reg

def train_snn(model, train_ld, val_ld, test_ld, n_epochs=80, lr=1e-3,
              device='cuda', patience=20, verbose_every=10):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val, best_state, wait = 0, None, 0
    t0 = time.time()
    for epoch in range(n_epochs):
        model.train()
        tot_loss, correct, total = 0, 0, 0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            logits, spks, _ = model(x)
            loss = criterion(logits, y)
            if spks: loss += spike_regularization(spks)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tot_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item(); total += len(y)
        val_acc = ev(model, val_ld, device, True)
        test_acc = ev(model, test_ld, device, True)
        if val_acc > best_val:
            best_val = val_acc
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
            wait = 0
        else: wait += 1
        if epoch % verbose_every == 0 or epoch == n_epochs-1:
            m = '*' if wait==0 else ''
            print(f'  Ep {epoch:3d}: loss={tot_loss/total:.4f} tr={correct/total:.4f} va={val_acc:.4f} te={test_acc:.4f} {m}')
        if wait >= patience: print(f'  Early stop at epoch {epoch}'); break
    if best_state:
        model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
    final = ev(model, test_ld, device, True)
    print(f'  Final: {final*100:.2f}% ({time.time()-t0:.0f}s)')
    return final, model

@torch.no_grad()
def ev(model, loader, device, is_snn=True):
    model.eval(); c, t = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if is_snn: logits, _, _ = model(x)
        else: logits = model(x)
        c += (logits.argmax(1)==y).sum().item(); t += len(y)
    return c/t

def train_baseline(model, tr, va, te, n_epochs=60, lr=1e-3, device='cuda', patience=20):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss(); best_val, best_state, wait = 0, None, 0; t0 = time.time()
    for ep in range(n_epochs):
        model.train()
        for x, y in tr:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y); opt.zero_grad(); loss.backward(); opt.step()
        va_a = ev(model, va, device, False)
        if va_a > best_val: best_val=va_a; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; wait=0
        else: wait += 1
        if ep % 20 == 0: print(f'  Ep {ep}: val={va_a:.4f}')
        if wait >= patience: print(f'  Early stop ep {ep}'); break
    if best_state: model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
    final = ev(model, te, device, False); print(f'  Final: {final*100:.2f}% ({time.time()-t0:.0f}s)')
    return final, model

## 4. Load FI-2010 Data

DeepLOB protocol: Days 1-7 train, Day 8 val, Day 9 test.
80/20 split from training for train/val.

In [7]:
fi_train_file = glob.glob('/kaggle/input/**/Train_Dst_NoAuction_DecPre_CF_7.txt', recursive=True)
assert fi_train_file, 'FI-2010 training file not found'
FI_DIR = os.path.dirname(fi_train_file[0])
print(f'FI_DIR: {FI_DIR}')

SEQ_LEN = 100; N_CLASSES = 3
def prep_x(d): return d[:40,:].T.astype(np.float32)
def get_lab(d): return (d[-5:,:].T.astype(int) - 1)
def make_seq(X, y, sl):
    n = len(X) - sl + 1
    return np.array([X[i:i+sl] for i in range(n)]).astype(np.float32), y[sl-1:]

train_raw = np.loadtxt(fi_train_file[0])
test8 = glob.glob('/kaggle/input/**/Test_Dst_NoAuction_DecPre_CF_8.txt', recursive=True)
test9 = glob.glob('/kaggle/input/**/Test_Dst_NoAuction_DecPre_CF_9.txt', recursive=True)
assert test8 and test9
test8_raw = np.loadtxt(test8[0]); test9_raw = np.loadtxt(test9[0])

# Full train sequences
X_full, y_full = make_seq(prep_x(train_raw), get_lab(train_raw)[:,0], SEQ_LEN)

# 80/20 split for train/val
val_split = int(len(X_full) * 0.8)
X_tr, y_tr = X_full[:val_split], y_full[:val_split]
X_va, y_va = X_full[val_split:], y_full[val_split:]
X_te, y_te = make_seq(prep_x(test9_raw), get_lab(test9_raw)[:,0], SEQ_LEN)

# Normalise
mu = X_tr.mean(axis=(0,1)); sd = X_tr.std(axis=(0,1)) + 1e-8
X_tr = (X_tr-mu)/sd; X_va = (X_va-mu)/sd; X_te = (X_te-mu)/sd

print(f'Train: {X_tr.shape}, Val: {X_va.shape}, Test: {X_te.shape}')
print(f'Labels (test): {np.bincount(y_te)}')

FI_DIR: /kaggle/input/datasets/ulfricirons/fi-2010/BenchmarkDatasets/NoAuction/3.NoAuction_DecPre/NoAuction_DecPre_Training


Train: (203720, 100, 40), Val: (50931, 100, 40), Test: (31838, 100, 40)
Labels (test): [ 5510 21311  5017]


In [8]:
class LOBDataset(Dataset):
    def __init__(self, X, y, encoding='direct', **kw):
        self.X, self.y = X, torch.tensor(y, dtype=torch.long)
        self.encoding, self.kw = encoding, kw
        self.input_size = self._enc(X[0]).shape[1]
    def _enc(self, s):
        if self.encoding == 'direct': return s
        elif self.encoding == 'delta': return SpikeEncoder.delta_modulation(s, **self.kw)
        elif self.encoding == 'adaptive_delta':
            e, _ = SpikeEncoder.adaptive_delta(s, **self.kw); return e
        elif self.encoding == 'log_direct': return SpikeEncoder.log_scale(s)
        return s
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        return torch.tensor(self._enc(self.X[i]), dtype=torch.float32), self.y[i]

BS = 256
# Direct loaders
train_ld = DataLoader(LOBDataset(X_tr, y_tr, 'direct'), BS, True, num_workers=2)
val_ld = DataLoader(LOBDataset(X_va, y_va, 'direct'), BS, False, num_workers=2)
test_ld = DataLoader(LOBDataset(X_te, y_te, 'direct'), BS, False, num_workers=2)
INPUT_SIZE = 40
print(f'Input: {INPUT_SIZE}, Steps: {SEQ_LEN}, Classes: {N_CLASSES}')

Input: 40, Steps: 100, Classes: 3


## 5. Baselines

In [9]:
print('--- LSTM ---')
model_lstm = LSTMBaseline(40, 128, 2, 3, 0.3)
acc_lstm, model_lstm = train_baseline(model_lstm, train_ld, val_ld, test_ld)
RESULTS['LSTM'] = acc_lstm

print('\n--- CNN ---')
model_cnn = CNNBaseline(40, 3)
acc_cnn, model_cnn = train_baseline(model_cnn, train_ld, val_ld, test_ld)
RESULTS['CNN'] = acc_cnn

--- LSTM ---


  Ep 0: val=0.6464


  Ep 20: val=0.7020


  Ep 40: val=0.6953


  Early stop ep 44


  Final: 76.73% (828s)

--- CNN ---


  Ep 0: val=0.6336


  Ep 20: val=0.6912


  Ep 40: val=0.6945


  Final: 75.02% (445s)


## 6. Base RSNN (Direct Input)

In [10]:
print('--- RSNN Direct (base) ---')
model_base = SNN(40, 256, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
print(f'Params: {model_base.count_params():,}')
acc_base, model_base = train_snn(model_base, train_ld, val_ld, test_ld)
RESULTS['RSNN_base'] = acc_base

--- RSNN Direct (base) ---
Params: 76,546


  Ep   0: loss=0.9360 tr=0.5976 va=0.6336 te=0.6694 *


  Ep  10: loss=0.9218 tr=0.5988 va=0.6336 te=0.6640 


  Ep  20: loss=0.9181 tr=0.5992 va=0.6336 te=0.6626 
  Early stop at epoch 20


  Final: 66.94% (3027s)


## 7. Exp F1: Adaptive Delta Encoding Sweep

Per-channel thresholds at percentile P of absolute inter-step changes.
Directly addresses the root cause: LOB changes are O(1e-4), fixed thresholds
produce zero spikes. Amir et al. 2017.

In [11]:
print('='*60)
print('F1: ADAPTIVE DELTA SWEEP')
print('='*60)

for P in [50, 75, 90, 95, 99]:
    print(f'\n--- Percentile={P} ---')
    ds_tr = LOBDataset(X_tr, y_tr, 'adaptive_delta', percentile=P)
    ds_va = LOBDataset(X_va, y_va, 'adaptive_delta', percentile=P)
    ds_te = LOBDataset(X_te, y_te, 'adaptive_delta', percentile=P)
    ld_tr = DataLoader(ds_tr, BS, True, num_workers=2)
    ld_va = DataLoader(ds_va, BS, False, num_workers=2)
    ld_te = DataLoader(ds_te, BS, False, num_workers=2)

    # Check spike density
    sample, _ = ds_tr[0]
    density = (sample.abs() > 0).float().mean().item()
    print(f'  Spike density: {density*100:.1f}%')

    m = SNN(ds_tr.input_size, 256, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
    acc, m = train_snn(m, ld_tr, ld_va, ld_te, n_epochs=60, patience=15)
    RESULTS[f'RSNN_adap_P{P}'] = acc

print('\nAdaptive delta results:')
for k,v in sorted(RESULTS.items()):
    if 'adap' in k: print(f'  {k}: {v*100:.2f}%')

F1: ADAPTIVE DELTA SWEEP

--- Percentile=50 ---
  Spike density: 41.3%


  Ep   0: loss=0.9669 tr=0.5918 va=0.6336 te=0.6693 *


  Ep  10: loss=0.9300 tr=0.5992 va=0.6327 te=0.6688 


  Ep  20: loss=0.9294 tr=0.5997 va=0.6330 te=0.6684 


  Early stop at epoch 24


...

  Ep   0: loss=0.9936 tr=0.5213 va=0.6220 te=0.6447 *


  Ep  10: loss=0.9388 tr=0.5950 va=0.6278 te=0.6586 


  Ep  20: loss=0.9253 tr=0.5985 va=0.6197 te=0.6562 


  Early stop at epoch 26


  Final: 65.98% (6574s)

Adaptive delta results:
  RSNN_adap_P50: 66.92%
  RSNN_adap_P75: 66.90%
  RSNN_adap_P90: 66.67%
  RSNN_adap_P95: 66.49%
  RSNN_adap_P99: 65.98%


## 8. Exp F2: Spike Activity Diagnosis

Quantify why the base RSNN fails: measure firing rates, active neurons,
and LOB data change magnitudes.

In [12]:
@torch.no_grad()
def spike_diagnosis(model, loader, device):
    model.eval()
    for x, y in loader:
        x = x.to(device)
        _, spks, _ = model(x)
        for i, s in enumerate(spks):
            B, T, N = s.shape
            total = B*T*N
            fired = s.sum().item()
            active = (s.sum(dim=1) > 0).float().mean(dim=0)
            print(f'  Layer {i}: sparsity={1-fired/total:.6f}, '
                  f'rate={fired/total:.6f}, '
                  f'active neurons={100*(active>0).float().mean().item():.1f}%')
        break

print('='*60)
print('F2: SPIKE ACTIVITY DIAGNOSIS')
print('='*60)

print('\nBase RSNN (direct input):')
spike_diagnosis(model_base, test_ld, device)

print('\nLOB data change magnitudes:')
sample = X_te[0]
diffs = np.abs(np.diff(sample, axis=0))
print(f'  Mean |diff|: {diffs.mean():.6f}')
print(f'  Max  |diff|: {diffs.max():.6f}')
print(f'  95th pct:    {np.percentile(diffs, 95):.6f}')
print(f'  99th pct:    {np.percentile(diffs, 99):.6f}')
print(f'  Fraction > 0.01: {(diffs > 0.01).mean()*100:.2f}%')
print(f'  Fraction > 0.1:  {(diffs > 0.1).mean()*100:.2f}%')
print(f'  Fraction > 1.0:  {(diffs > 1.0).mean()*100:.2f}%')

F2: SPIKE ACTIVITY DIAGNOSIS

Base RSNN (direct input):


  Layer 0: sparsity=0.816143, rate=0.183857, active neurons=32.0%



LOB data change magnitudes:
  Mean |diff|: 0.037652
  Max  |diff|: 0.618415
  95th pct:    0.226377
  99th pct:    0.367463
  Fraction > 0.01: 26.24%
  Fraction > 0.1:  15.45%
  Fraction > 1.0:  0.00%


## 9. Exp F3: Log-Scaled Input

Apply sign(x) * log(1+|x|) before feeding into LIF. Amplifies small changes.
Not a new encoding — a preprocessing step before direct input.

In [13]:
print('='*60)
print('F3: LOG-SCALED INPUT')
print('='*60)

ds_tr_log = LOBDataset(X_tr, y_tr, 'log_direct')
ds_va_log = LOBDataset(X_va, y_va, 'log_direct')
ds_te_log = LOBDataset(X_te, y_te, 'log_direct')
ld_tr_log = DataLoader(ds_tr_log, BS, True, num_workers=2)
ld_va_log = DataLoader(ds_va_log, BS, False, num_workers=2)
ld_te_log = DataLoader(ds_te_log, BS, False, num_workers=2)

m_log = SNN(40, 256, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
acc_log, m_log = train_snn(m_log, ld_tr_log, ld_va_log, ld_te_log)
RESULTS['RSNN_log_scale'] = acc_log

F3: LOG-SCALED INPUT


  Ep   0: loss=0.9369 tr=0.5978 va=0.6336 te=0.6695 *


  Ep  10: loss=0.9297 tr=0.5985 va=0.6336 te=0.6694 


  Ep  20: loss=0.9261 tr=0.5989 va=0.6336 te=0.6684 
  Early stop at epoch 20


  Final: 66.95% (3024s)


## 10. Exp F4: Time Constant Sweep

LOB events are fast. Short tau (5-10ms) may work better than the default 20ms.

In [14]:
print('='*60)
print('F4: TAU SWEEP')
print('='*60)

for tau in [5.0, 10.0, 50.0, 100.0]:
    print(f'\n--- tau_mem={tau} ---')
    m = SNN(40, 256, 3, 1, True, tau, tau/2, 10.0, True, 'max_over_time', 0.3)
    acc, _ = train_snn(m, train_ld, val_ld, test_ld, n_epochs=60, patience=15)
    RESULTS[f'RSNN_tau{int(tau)}'] = acc

F4: TAU SWEEP

--- tau_mem=5.0 ---


  Ep   0: loss=0.9380 tr=0.5971 va=0.6336 te=0.6698 *


  Ep  10: loss=0.9302 tr=0.5981 va=0.6336 te=0.6683 


  Early stop at epoch 15


  Final: 66.98% (2282s)

--- tau_mem=10.0 ---


...

  Final: 66.94% (2267s)

--- tau_mem=100.0 ---


  Ep   0: loss=0.9345 tr=0.5976 va=0.6336 te=0.6694 *


  Ep  10: loss=0.9187 tr=0.5987 va=0.6336 te=0.6679 


  Early stop at epoch 15


  Final: 66.94% (2256s)


## 11. Exp F5: Hidden Size Sweep

With 40 inputs, 512 neurons may overfit. 64-128 might be more appropriate.

In [15]:
print('='*60)
print('F5: HIDDEN SIZE SWEEP')
print('='*60)

for hs in [64, 128, 512]:
    print(f'\n--- hidden_size={hs} ---')
    m = SNN(40, hs, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
    print(f'Params: {m.count_params():,}')
    acc, _ = train_snn(m, train_ld, val_ld, test_ld, n_epochs=60, patience=15)
    RESULTS[f'RSNN_h{hs}'] = acc

F5: HIDDEN SIZE SWEEP

--- hidden_size=64 ---
Params: 6,850


  Ep   0: loss=0.9389 tr=0.5975 va=0.6336 te=0.6694 *


## 12. Exp F6: Input Batch Normalisation

LOB features drift within the 100-event window. BN normalises per feature
across the batch, stabilising LIF threshold consistency.
Kim & Panda 2021, Frontiers Neurosci.

In [ ]:
print('='*60)
print('F6: INPUT BATCH NORM')
print('='*60)

class SNN_BN(SNN):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Get input_size from first hidden layer
        self.bn = nn.BatchNorm1d(self.hidden_layers[0].W_ff.in_features)
    def forward(self, x):
        B, T, C = x.shape
        x = self.bn(x.reshape(B*T, C)).reshape(B, T, C)
        return super().forward(x)

m_bn = SNN_BN(40, 256, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
acc_bn, m_bn = train_snn(m_bn, train_ld, val_ld, test_ld)
RESULTS['RSNN_input_bn'] = acc_bn

## 12b. Temporal Analysis

Running-max and instantaneous readout accuracy at each timestep.
Compare RSNN vs LSTM vs CNN.

In [ ]:
@torch.no_grad()
def temporal_rsnn(model, loader, device):
    model.eval()
    all_c = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        _, _, om = model(x)
        B, T, C = om.shape
        rmax = torch.full((B, C), -float('inf'), device=device)
        ct = torch.zeros(T, device=device)
        for t in range(T):
            rmax = torch.max(rmax, om[:, t])
            ct[t] += (rmax.argmax(1) == y).float().sum()
        all_c.append(ct)
    total = torch.stack(all_c).sum(0)
    n = sum(len(y) for _, y in loader)
    return (total / n).cpu().numpy()

@torch.no_grad()
def temporal_lstm(model, loader, device):
    model.eval()
    all_c = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        hseq = model.forward_seq(x)
        lseq = model.fc(hseq)
        B, T, C = lseq.shape
        rmax = torch.full((B, C), -float('inf'), device=device)
        ct = torch.zeros(T, device=device)
        for t in range(T):
            rmax = torch.max(rmax, lseq[:, t])
            ct[t] += (rmax.argmax(1) == y).float().sum()
        all_c.append(ct)
    total = torch.stack(all_c).sum(0)
    n = sum(len(y) for _, y in loader)
    return (total / n).cpu().numpy()

@torch.no_grad()
def temporal_instant(model, loader, device, is_lstm=False):
    model.eval()
    all_c = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if is_lstm:
            seq = model.fc(model.forward_seq(x))
        else:
            _, _, seq = model(x)
        B, T, C = seq.shape
        ct = torch.zeros(T, device=device)
        for t in range(T):
            ct[t] += (seq[:, t].argmax(1) == y).float().sum()
        all_c.append(ct)
    total = torch.stack(all_c).sum(0)
    n = sum(len(y) for _, y in loader)
    return (total / n).cpu().numpy()

print('='*60)
print('TEMPORAL ANALYSIS')
print('='*60)

rsnn_rm = temporal_rsnn(model_base, test_ld, device)
lstm_rm = temporal_lstm(model_lstm, test_ld, device)
rsnn_inst = temporal_instant(model_base, test_ld, device, False)
lstm_inst = temporal_instant(model_lstm, test_ld, device, True)

steps = np.arange(SEQ_LEN)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(steps, rsnn_rm*100, lw=2, color='steelblue', label=f'RSNN ({acc_base*100:.1f}%)')
axes[0].plot(steps, lstm_rm*100, lw=2, color='coral', label=f'LSTM ({acc_lstm*100:.1f}%)')
axes[0].axhline(y=acc_cnn*100, color='green', ls='--', lw=1.5, label=f'CNN ({acc_cnn*100:.1f}%)')
axes[0].set_xlabel('LOB events'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Running-max readout'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, rsnn_inst*100, lw=2, color='steelblue', label='RSNN instant')
axes[1].plot(steps, lstm_inst*100, lw=2, color='coral', label='LSTM instant')
axes[1].set_xlabel('LOB events'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Instantaneous readout'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'fi_temporal.png'), dpi=150); plt.show()

## 13. Analysis Suite (F7-F10)

In [ ]:
# ========== F7: NOISE ROBUSTNESS ==========
print('='*60)
print('F7: NOISE ROBUSTNESS')
print('='*60)

@torch.no_grad()
def eval_noise(model, loader, device, sigma, is_snn=True):
    model.eval(); c, t = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        xn = x + sigma*torch.randn_like(x)
        if is_snn: logits, _, _ = model(xn)
        else: logits = model(xn)
        c += (logits.argmax(1)==y).sum().item(); t += len(y)
    return c/t

sigmas = [0, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
rn, ln = [], []
for s in sigmas:
    r = eval_noise(model_base, test_ld, device, s, True)
    l = eval_noise(model_lstm, test_ld, device, s, False)
    rn.append(r); ln.append(l)
    print(f'  sigma={s:.2f}: RSNN={r*100:.1f}% LSTM={l*100:.1f}% gap={100*(r-l):+.1f}pp')

In [ ]:
# ========== F8: SPIKE EFFICIENCY ==========
print('\n' + '='*60)
print('F8: SPIKE EFFICIENCY')
print('='*60)

@torch.no_grad()
def count_spikes(model, loader, device):
    model.eval(); counts = []
    for x, y in loader:
        x = x.to(device); _, spks, _ = model(x)
        batch_total = sum(s.sum(dim=(1,2)).cpu().numpy() for s in spks)
        counts.append(batch_total)
    return np.concatenate(counts)

sc = count_spikes(model_base, test_ld, device)
rsnn_ops = sc.mean() * (256 + 3)
lstm_ops = (4*(40*128+128*128) + 4*(128*128+128*128)) * SEQ_LEN
print(f'RSNN: mean {sc.mean():.0f} spikes, {rsnn_ops:,.0f} ops')
print(f'LSTM: {lstm_ops:,.0f} ops ({lstm_ops/max(rsnn_ops,1):.1f}x more)')

In [ ]:
# ========== F9: RECURRENCE ABLATION ==========
print('\n' + '='*60)
print('F9: RECURRENCE ABLATION')
print('='*60)

model_abl = copy.deepcopy(model_base)
with torch.no_grad():
    for layer in model_abl.hidden_layers:
        if hasattr(layer, 'W_rec'): layer.W_rec.weight.data.zero_()
acc_abl = ev(model_abl, test_ld, device, True)
print(f'With recurrence:    {acc_base*100:.2f}%')
print(f'Without:            {acc_abl*100:.2f}%')
print(f'Drop: {(acc_base-acc_abl)*100:.2f}pp')

In [ ]:
# ========== F10: EARLY CLASSIFICATION ==========
print('\n' + '='*60)
print('F10: EARLY CLASSIFICATION')
print('='*60)

@torch.no_grad()
def early_class(model, loader, device, is_lstm=False):
    model.eval(); all_fc, all_lab = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device); B = x.shape[0]
        if is_lstm: lseq = model.fc(model.forward_seq(x))
        else: _, _, lseq = model(x)
        _, T, C = lseq.shape
        rmax = torch.full((B, C), -float('inf'), device=device)
        fc = torch.full((B,), -1, dtype=torch.long, device=device)
        for t in range(T):
            rmax = torch.max(rmax, lseq[:, t])
            newly = (rmax.argmax(1) == y) & (fc == -1); fc[newly] = t
        all_fc.append(fc.cpu()); all_lab.append(y.cpu())
    return torch.cat(all_fc).numpy(), torch.cat(all_lab).numpy()

rfc, labels = early_class(model_base, test_ld, device, False)
lfc, _ = early_class(model_lstm, test_ld, device, True)
CLASS_NAMES = ['Down', 'Stationary', 'Up']

for c in range(3):
    m = labels == c
    rv = rfc[m][rfc[m]>=0]; lv = lfc[m][lfc[m]>=0]
    rm = np.median(rv) if len(rv)>0 else float('inf')
    lm = np.median(lv) if len(lv)>0 else float('inf')
    print(f'  {CLASS_NAMES[c]:>12}: RSNN med={rm:.0f} steps, LSTM med={lm:.0f} steps')

## 14. Exp F11: Multi-Horizon Analysis

Longer horizons (k=50, k=100) have larger mid-price movements.
If the RSNN gap narrows, the issue is signal magnitude.

In [ ]:
print('='*60)
print('F11: MULTI-HORIZON')
print('='*60)

y_full_all = get_lab(train_raw)  # all 5 horizons
y_te_all = get_lab(test9_raw)

HORIZONS = {0: 'k=10', 2: 'k=30', 4: 'k=100'}  # indices into 5 horizons

for h_idx, h_name in HORIZONS.items():
    print(f'\n--- {h_name} ---')
    _, y_tr_h = make_seq(prep_x(train_raw), get_lab(train_raw)[:, h_idx], SEQ_LEN)
    y_tr_h = y_tr_h[:val_split]; y_va_h = y_full_all[val_split+SEQ_LEN-1:, h_idx]
    # Fix: recompute val labels properly
    _, y_full_h = make_seq(prep_x(train_raw), get_lab(train_raw)[:, h_idx], SEQ_LEN)
    y_tr_h = y_full_h[:val_split]
    y_va_h = y_full_h[val_split:]
    _, y_te_h = make_seq(prep_x(test9_raw), get_lab(test9_raw)[:, h_idx], SEQ_LEN)

    # Reuse X sequences, just change labels
    ds_tr_h = LOBDataset(X_tr, y_tr_h, 'direct')
    ds_va_h = LOBDataset(X_va, y_va_h, 'direct')
    ds_te_h = LOBDataset(X_te, y_te_h, 'direct')
    ld_tr_h = DataLoader(ds_tr_h, BS, True, num_workers=2)
    ld_va_h = DataLoader(ds_va_h, BS, False, num_workers=2)
    ld_te_h = DataLoader(ds_te_h, BS, False, num_workers=2)

    m_r = SNN(40, 256, 3, 1, True, 20.0, 10.0, 10.0, True, 'max_over_time', 0.3)
    acc_r, _ = train_snn(m_r, ld_tr_h, ld_va_h, ld_te_h, n_epochs=60, patience=15)

    m_l = LSTMBaseline(40, 128, 2, 3, 0.3)
    acc_l, _ = train_baseline(m_l, ld_tr_h, ld_va_h, ld_te_h)

    RESULTS[f'RSNN_{h_name}'] = acc_r
    RESULTS[f'LSTM_{h_name}'] = acc_l
    print(f'  RSNN: {acc_r*100:.2f}%, LSTM: {acc_l*100:.2f}%, gap: {(acc_r-acc_l)*100:+.2f}pp')

## 15. Exp F12: Feature Importance (Price vs Volume)

Zero out price features (20) or volume features (20) at test time.
FI-2010: 40 features = 10 levels x (ask_price, ask_vol, bid_price, bid_vol).
Price = even indices, Volume = odd indices.

In [ ]:
print('='*60)
print('F12: PRICE vs VOLUME')
print('='*60)
print(f'Baseline: {acc_base*100:.2f}%\n')

price_idx = list(range(0, 40, 2))  # ask_price, bid_price
vol_idx = list(range(1, 40, 2))    # ask_vol, bid_vol

@torch.no_grad()
def eval_zeroed(model, loader, device, zero_idx):
    model.eval(); c, t = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        x_z = x.clone(); x_z[:, :, zero_idx] = 0.0
        logits, _, _ = model(x_z)
        c += (logits.argmax(1)==y).sum().item(); t += len(y)
    return c/t

acc_no_price = eval_zeroed(model_base, test_ld, device, price_idx)
acc_no_vol = eval_zeroed(model_base, test_ld, device, vol_idx)
print(f'Prices zeroed: {acc_no_price*100:.2f}% (drop: {(acc_base-acc_no_price)*100:.2f}pp)')
print(f'Volumes zeroed: {acc_no_vol*100:.2f}% (drop: {(acc_base-acc_no_vol)*100:.2f}pp)')
print(f'\nRSNN relies more on: {"prices" if acc_no_price < acc_no_vol else "volumes"}')

## 16. Exp F13: Per-Level Importance

Zero out level 1, levels 1-2, levels 1-5, levels 1-10.
Best LOB information is at the top of the book (level 1).

In [ ]:
print('='*60)
print('F13: PER-LEVEL IMPORTANCE')
print('='*60)
print(f'Baseline: {acc_base*100:.2f}%\n')

# Each level = 4 features: ask_price, ask_vol, bid_price, bid_vol
for n_levels in [1, 2, 5, 10]:
    zero_idx = list(range(n_levels * 4))  # zero out first n_levels
    acc = eval_zeroed(model_base, test_ld, device, zero_idx)
    print(f'  Levels 1-{n_levels} zeroed ({n_levels*4} features): '
          f'{acc*100:.2f}% (drop: {(acc_base-acc)*100:.2f}pp)')

## 17. Complete Results

In [ ]:
print('='*70)
print('FI-2010 COMPLETE RESULTS')
print('='*70)

for k, v in sorted(RESULTS.items(), key=lambda x: x[1], reverse=True):
    print(f'  {k:30s} {v*100:7.2f}%')

print(f'\n--- Key findings ---')
print(f'Base RSNN (direct): {acc_base*100:.2f}%')
print(f'LSTM: {acc_lstm*100:.2f}% | CNN: {acc_cnn*100:.2f}%')
print(f'Gap: {(acc_base-acc_lstm)*100:.2f}pp')
best_adap = max((v for k,v in RESULTS.items() if 'adap' in k), default=0)
print(f'Best adaptive delta: {best_adap*100:.2f}%')
print(f'Recurrence ablation: {acc_base*100:.2f}% -> {acc_abl*100:.2f}%')

with open(os.path.join(SAVE_DIR, 'fi2010_extended_results.json'), 'w') as f:
    json.dump(RESULTS, f, indent=2, default=str)
print('\nSaved fi2010_extended_results.json')